<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/12%20-%20Matrizes%20de%20Incidencia%20Adjacencia%20e%20Custos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 12 - Notebook: Matriz de Incidência e Balanço de Fluxo de Peças
## Célula de Manufatura Flexível (FMS)

Neste notebook construímos a **Matriz de Incidência Vértice-Aresta** $B \in \{-1, 0, 1\}^{n \times m}$, verificamos a propriedade formal da soma nula por coluna e resolvemos o balanço de fluxo contínuo de materiais $B \cdot \vec{Q} = \vec{S}$ na malha de esteiras e triagem pneumática.

In [ ]:
from typing import List, Dict, Tuple, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[Any]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura alinhada."""
    larguras = [max(len(str(r)), 7) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoManufatura:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_rota(self, origem: str, destino: str, comprimento_m: float,
                        tag_atuador: str, mecanismo: str = "Esteira"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador ISA": tag_atuador,
            "Mecanismo": mecanismo
        })

def criar_planta_fms() -> GrafoManufatura:
    """Instancia a topologia padrão da Célula de Manufatura Flexível."""
    postos = [
        "SILO-100", "INSP-200", "DIV-201", "DIV-202", "DIV-203",
        "CX-301", "CX-302", "CX-303", "DESC-304"
    ]
    g = GrafoManufatura(postos)
    g.adicionar_rota("SILO-100", "INSP-200", 0.8, "XV-101", "Alimentador")
    g.adicionar_rota("INSP-200", "DIV-201",  1.2, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-201",  "CX-301",   0.5, "XV-201", "Pistão 1")
    g.adicionar_rota("DIV-201",  "DIV-202",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-202",  "CX-302",   0.5, "XV-202", "Pistão 2")
    g.adicionar_rota("DIV-202",  "DIV-203",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-203",  "CX-303",   0.5, "XV-203", "Pistão 3")
    g.adicionar_rota("DIV-203",  "DESC-304", 0.8, "M-101",  "Calha Final")
    return g

fms = criar_planta_fms()

class CalculadorIncidenciaFMS:
    @staticmethod
    def construir_matriz_incidencia(vertices: List[str], arestas: List[Dict[str, Any]]) -> Tuple[List[List[int]], List[str]]:
        """
        Gera a Matriz de Incidência Vértice-Aresta B in {-1, 0, 1}^(n x m).
        B[i, j] = -1 se a aresta j sai do vértice i.
        B[i, j] = +1 se a aresta j entra no vértice i.
        """
        n = len(vertices)
        m = len(arestas)
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * m for _ in range(n)]
        rotulos_e = []
        for j, a in enumerate(arestas):
            u = v_idx[a["Origem"]]
            v = v_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
            rotulos_e.append(f"e{j+1}:{a['Origem']}->{a['Destino']}")
        return B, rotulos_e

# 1. Construção da Matriz de Incidência B
B_mat, rotulos_arestas = CalculadorIncidenciaFMS.construir_matriz_incidencia(fms.vertices, fms.arestas_detalhes)
nomes_colunas = [f"e{j+1}" for j in range(len(fms.arestas_detalhes))]

print("=== MATRIZ DE INCIDÊNCIA B (POSTOS X ESTEIRAS) ===")
print(formatar_matriz(B_mat, fms.vertices, nomes_colunas))

# 2. Verificação Formal: Soma por coluna deve ser identicamente nula
somas_colunas = [sum(B_mat[i][j] for i in range(len(fms.vertices))) for j in range(len(fms.arestas_detalhes))]
assert all(s == 0 for s in somas_colunas), "Erro: Propriedade formal de soma nula violada na Matriz B!"

# 3. Simulação de Balanço Contínuo de Peças: S = B * Q
# Cenário de operação em regime estacionário:
# Taxa de entrada: 20 peças/min alimentadas pelo Silo (e1 = 20.0)
# Distribuição na triagem:
# e1 (SILO->INSP) = 20.0
# e2 (INSP->DIV1) = 20.0
# e3 (DIV1->CX1, Vermelhas) = 8.0 peças/min
# e4 (DIV1->DIV2) = 12.0 peças/min
# e5 (DIV2->CX2, Verdes) = 6.0 peças/min
# e6 (DIV2->DIV3) = 6.0 peças/min
# e7 (DIV3->CX3, Azuis) = 5.0 peças/min
# e8 (DIV3->DESC, Descarte) = 1.0 peça/min
Q_vec = [20.0, 20.0, 8.0, 12.0, 6.0, 6.0, 5.0, 1.0]

S_res = []
for i, v_nome in enumerate(fms.vertices):
    balanco = sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec)))
    S_res.append({"Posto de Trabalho": v_nome, "Balanço Líquido (peças/min)": f"{balanco:.1f}"})

print("\n--- BALANÇO DE FLUXO DE PEÇAS EM REGIME PERMANENTE (S = B * Q) ---")
print(formatar_tabela(S_res))

# Validações determinísticas
idx_silo = fms.v_to_idx["SILO-100"]
idx_cx1  = fms.v_to_idx["CX-301"]
idx_insp = fms.v_to_idx["INSP-200"]

# O Silo deve ser fonte pura (-20 peças/min)
assert sum(B_mat[idx_silo][j] * Q_vec[j] for j in range(len(Q_vec))) == -20.0
# Os postos intermediários de esteira devem ter saldo nulo (sem entupimento)
assert sum(B_mat[idx_insp][j] * Q_vec[j] for j in range(len(Q_vec))) == 0.0
# As caixas finais devem somar exatamente a taxa de infeed: 8 + 6 + 5 + 1 = 20 peças/min
taxa_total_saidas = sum(sum(B_mat[fms.v_to_idx[cx]][j] * Q_vec[j] for j in range(len(Q_vec))) for cx in ["CX-301", "CX-302", "CX-303", "DESC-304"])
assert taxa_total_saidas == 20.0

print("\n[OK] Matriz de Incidência e Conservação de Fluxo da FMS validadas com sucesso!")

=== MATRIZ DE INCIDÊNCIA B (POSTOS X ESTEIRAS) ===
         |      e1 |      e2 |      e3 |      e4 |      e5 |      e6 |      e7 |      e8
---------+---------+---------+---------+---------+---------+---------+---------+--------
SILO-100 |      -1 |       0 |       0 |       0 |       0 |       0 |       0 |       0
INSP-200 |       1 |      -1 |       0 |       0 |       0 |       0 |       0 |       0
DIV-201  |       0 |       1 |      -1 |      -1 |       0 |       0 |       0 |       0
DIV-202  |       0 |       0 |       0 |       1 |      -1 |      -1 |       0 |       0
DIV-203  |       0 |       0 |       0 |       0 |       0 |       1 |      -1 |      -1
CX-301   |       0 |       0 |       1 |       0 |       0 |       0 |       0 |       0
CX-302   |       0 |       0 |       0 |       0 |       1 |       0 |       0 |       0
CX-303   |       0 |       0 |       0 |       0 |       0 |       0 |       1 |       0
DESC-304 |       0 |       0 |       0 |       0 |       0 